# Lawha لوحة — Arabic licence plate recognition

Two models, trained and evaluated end to end on the Egyptian EALPR dataset:

1. **Detector**: a pretrained YOLO nano model fine-tuned to find plates in vehicle photos.
2. **Reader**: a small CRNN trained from scratch with CTC loss to read the Arabic letters and digits on the cropped plate.

Code: https://github.com/readaloush/lawha

**Settings:** turn on *Internet* and pick a *GPU* accelerator in the notebook sidebar before running.

In [ ]:
!pip -q install ultralytics onnx onnxruntime
!git clone -q https://github.com/readaloush/lawha.git
!git clone -q --depth 1 https://github.com/ahmedramadan96/EALPR.git lawha/data/EALPR
%cd lawha
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 1. Data

EALPR links three things by vehicle id: the photo, the plate crop and the character boxes. Every split is made **by vehicle id** (70/15/15), so a plate never appears in training while its photo sits in the test set.

In [ ]:
from lawha.data import load_plates, split_ids
plates = load_plates('data/EALPR'); split = split_ids('data/EALPR')
print(len(plates), 'labelled plates;', {k: len(v) for k, v in split.items()}, 'vehicles')
!python -m lawha.detector prepare --data data/EALPR --out data/yolo

## 2. Train the plate reader (CRNN + CTC)

In [ ]:
!python -m lawha.train_reader --data data/EALPR --epochs 80

## 3. Train the plate detector (YOLO nano, fine-tuned)

In [ ]:
!python -m lawha.detector train --yolo data/yolo --epochs 60 --imgsz 640 --device 0 2>&1 | tail -40

## 4. Evaluate the whole pipeline on the test vehicles

A plate counts as correct only if the detector found it **and** the reader got every character right.

In [ ]:
!python -m lawha.pipeline --data data/EALPR

## 5. Figures

In [ ]:
!python -m lawha.report --data data/EALPR
from IPython.display import Image, display
for f in ['results/reader_curves.png', 'results/pipeline_summary.png', 'results/examples.png']:
    display(Image(f))

## 6. Export for deployment and collect outputs

In [ ]:
!python -m lawha.export
!cd results && zip -qr /kaggle/working/lawha_results.zip . -x 'runs/detector/weights/last.pt'
!ls -la /kaggle/working/lawha_results.zip
# keep only figures, metrics and exported models in the notebook output
!mkdir -p /kaggle/working/keep && cd results && rm -f pipeline_errors.json && cp *.png *.json *.onnx reader.pt /kaggle/working/keep/
%cd /kaggle/working
!rm -rf /kaggle/working/lawha /kaggle/working/lawha_results.zip && ls -la /kaggle/working /kaggle/working/keep